<a href="https://colab.research.google.com/github/juanpajaro/IA_en_salud_diplomado_puj/blob/main/Clase_6_clasificacion_arboles_crossv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Importamos librerias**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV, KFold
from xgboost import XGBClassifier

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
import shap

#**Cargamos datos**

In [ ]:
data = pd.read_csv("heart.csv")

##**Ajustamos datos**

In [ ]:
data = data[data['ca'] < 4] #drop the wrong ca values
data = data[data['thal'] > 0] # drop the wong thal value
print(f'The length of the data now is {len(data)} instead of 303!')


data = data.rename(
    columns = {'cp':'chest_pain_type',
               'trestbps':'resting_blood_pressure',
               'chol': 'cholesterol',
               'fbs': 'fasting_blood_sugar',
               'restecg' : 'resting_electrocardiogram',
               'thalach': 'max_heart_rate_achieved',
               'exang': 'exercise_induced_angina',
               'oldpeak': 'st_depression',
               'slope': 'st_slope',
               'ca':'num_major_vessels',
               'thal': 'thalassemia'},
    errors="raise")

data['sex'][data['sex'] == 0] = 'female'
data['sex'][data['sex'] == 1] = 'male'

data['chest_pain_type'][data['chest_pain_type'] == 0] = 'typical angina'
data['chest_pain_type'][data['chest_pain_type'] == 1] = 'atypical angina'
data['chest_pain_type'][data['chest_pain_type'] == 2] = 'non-anginal pain'
data['chest_pain_type'][data['chest_pain_type'] == 3] = 'asymptomatic'

data['fasting_blood_sugar'][data['fasting_blood_sugar'] == 0] = 'lower than 120mg/ml'
data['fasting_blood_sugar'][data['fasting_blood_sugar'] == 1] = 'greater than 120mg/ml'

data['resting_electrocardiogram'][data['resting_electrocardiogram'] == 0] = 'normal'
data['resting_electrocardiogram'][data['resting_electrocardiogram'] == 1] = 'ST-T wave abnormality'
data['resting_electrocardiogram'][data['resting_electrocardiogram'] == 2] = 'left ventricular hypertrophy'

data['exercise_induced_angina'][data['exercise_induced_angina'] == 0] = 'no'
data['exercise_induced_angina'][data['exercise_induced_angina'] == 1] = 'yes'

data['st_slope'][data['st_slope'] == 0] = 'upsloping'
data['st_slope'][data['st_slope'] == 1] = 'flat'
data['st_slope'][data['st_slope'] == 2] = 'downsloping'

data['thalassemia'][data['thalassemia'] == 1] = 'fixed defect'
data['thalassemia'][data['thalassemia'] == 2] = 'normal'
data['thalassemia'][data['thalassemia'] == 3] = 'reversable defect'

In [ ]:
# numerical fearures 6
num_feats = ['age', 'cholesterol', 'resting_blood_pressure', 'max_heart_rate_achieved', 'st_depression', 'num_major_vessels']
# categorical (binary)
bin_feats = ['sex', 'fasting_blood_sugar', 'exercise_induced_angina', 'target']
# caterorical (multi-)
nom_feats= ['chest_pain_type', 'resting_electrocardiogram', 'st_slope', 'thalassemia']
cat_feats = nom_feats + bin_feats


#**Analisis de datos**

In [ ]:
data.head()

In [ ]:
mypal= ['#FC05FB', '#FEAEFE', '#FCD2FC','#F3FEFA', '#B4FFE4','#3FFEBA']

plt.figure(figsize=(7, 5),facecolor='#F6F5F4')
total = float(len(data))
ax = sns.countplot(x=data['target'], palette=mypal[1::4])
ax.set_facecolor('#F6F5F4')

for p in ax.patches:

    height = p.get_height()
    ax.text(p.get_x()+p.get_width()/2.,height + 3,'{:1.1f} %'.format((height/total)*100), ha="center",
           bbox=dict(facecolor='none', edgecolor='black', boxstyle='round', linewidth=0.5))

ax.set_title('Target variable distribution', fontsize=20, y=1.05)
sns.despine(right=True)
sns.despine(offset=5, trim=True)

#**Codificacion de datos**

In [ ]:
def label_encode_cat_features(data, cat_features):
    '''
    Given a dataframe and its categorical features, this function returns label-encoded dataframe
    '''

    label_encoder = LabelEncoder()
    data_encoded = data.copy()

    for col in cat_features:
        data_encoded[col] = label_encoder.fit_transform(data[col])

    data = data_encoded

    return data


In [ ]:
cat_features = cat_feats
data = label_encode_cat_features(data, cat_features)

seed = 48
features = data.columns[:-1]

X = data[features]
y = data['target']

#**Seleccionamos y cargamos el algoritmo**

In [ ]:
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      random_state=seed)

##**Estrategia de validación cruzada (cross-validation)**

In [ ]:
kf = KFold(n_splits=None, shuffle=True, random_state=seed)

##**Busqueda de grilla de hiperparametros**

In [ ]:
# Probaremos diferentes combinaciones de profundidad, tasa de aprendizaje y número de árboles
param_grid = {
    'max_depth': None,
    'learning_rate': None,
    'n_estimators': None,
}

#Configurar GridSearchCV
# 'refit="f1"' asegura que al final, el mejor modelo se elija basándose en el F1-Score
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=kf,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    refit='f1',
    n_jobs=-1, # Usa todos los núcleos de tu procesador para ir más rápido
    verbose=1
)

##**Entrenamos el algoritmo**

In [ ]:
grid_search.fit(X, y)

In [ ]:
print(f"\n--- Mejores Hiperparámetros encontrados ---")
print(grid_search.best_params_)

#**Evaluacion**

In [ ]:
print(f"\n--- Métricas del mejor modelo (Basado en F1-Score) ---")
best_index = grid_search.best_index_
print(f"Exactitud:    {grid_search.cv_results_['mean_test_accuracy'][best_index]:.4f}")
print(f"Precisión:    {grid_search.cv_results_['mean_test_precision'][best_index]:.4f}")
print(f"Sensibilidad: {grid_search.cv_results_['mean_test_recall'][best_index]:.4f}")
print(f"F1-Score:     {grid_search.cv_results_['mean_test_f1'][best_index]:.4f}")